# Statsmodels OLS 极速集训营 🔥
本节专为破解「如何使用 Patsy 语法书写控制变量和交互项」而生，助你在最短时间内掌握高级 DA 在业务推断或实验中的因果回归核心手感。

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# 直接拉取线上清洗好格式的原始数据进行 OLS 练习
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df.columns = df.columns.str.lower()

# 简单预处理（与上一节保持一致，清洗空值）
df['totalcharges'] = pd.to_numeric(df['totalcharges'], errors='coerce')
df = df.dropna()

df.head(3)

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import platform

# 1. 忽略烦人的警告
warnings.filterwarnings('ignore')

# 2. Pandas 显示设置 (破解行列显示限制)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.3f' % x) # 避免科学计数法

# 2. 绘图设置 (高清 + 样式)
%matplotlib inline
sns.set(style='whitegrid', palette='muted', font_scale=1.5)

# 3. 解决中文乱码 (Mac/Windows 自适应) 🇨🇳
if platform.system() == 'Darwin':
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS'] # Mac专用
else:
    plt.rcParams['font.sans-serif'] = ['SimHei'] # Windows专用

plt.rcParams['axes.unicode_minus'] = False # 解决负号显示问题

In [3]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerid,7032,7032,7590-VHVEG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7032,2,Male,3549,NaN,NaN,NaN,NaN,NaN,NaN,NaN
seniorcitizen,7032.000,NaN,NaN,NaN,0.162,0.369,0.000,0.000,0.000,0.000,1.000
partner,7032,2,No,3639,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dependents,7032,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7032.000,NaN,NaN,NaN,32.422,24.545,1.000,9.000,29.000,55.000,72.000
phoneservice,7032,2,Yes,6352,NaN,NaN,NaN,NaN,NaN,NaN,NaN
multiplelines,7032,3,No,3385,NaN,NaN,NaN,NaN,NaN,NaN,NaN
internetservice,7032,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
onlinesecurity,7032,3,No,3497,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 第一关（语法关）：分类变量的正确处理
写一行代码，只拟合普通 OLS 回归。把 `df` 里影响 **月费 (`monthlycharges`)** 的几个关键因子放进一个模型：
必须要包含 `gender`（分类变量），`dependents`（家属情况/分类变量），还需要带上 `tenure`（在网时间/连续变量）。
**任务：** 拟合模型并打印它的 `summary()`。

In [4]:
# [你的代码区] 语法关：因变量为 monthlycharges
formula_1 = 'monthlycharges ~ C(gender) + C(dependents) + tenure'
model_1 = smf.ols(formula_1, data=df).fit()
print(model_1.summary())

                            OLS Regression Results                            
Dep. Variable:         monthlycharges   R-squared:                       0.085
Model:                            OLS   Adj. R-squared:                  0.085
Method:                 Least Squares   F-statistic:                     217.8
Date:                Tue, 24 Feb 2026   Prob (F-statistic):          3.98e-135
Time:                        15:47:28   Log-Likelihood:                -33602.
No. Observations:                7032   AIC:                         6.721e+04
Df Residuals:                    7028   BIC:                         6.724e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               57.4559 

## 第二关（数值魔法关）：直接在公式里做非线性转换
假设 `tenure` （在网月数）和月费不是直线关系，而是“随着时间流逝，按对数规律平缓增长”。
**任务：** Statsmodels 允许你在表达式里直接写 Numpy 的特征工程计算式。请写出带有 `tenure` 取对数 (`np.log()`) 项的回归公式，依然包含性别和家属情况，去预测 `monthlycharges`。

In [5]:
# [你的代码区] 魔法关：直接在 formula 里写 Numpy 函数
formula_2 = 'monthlycharges ~ C(gender) + C(dependents) + np.log(tenure)'

model_2 = smf.ols(formula_2,data=df).fit()
print(model_2.summary())


                            OLS Regression Results                            
Dep. Variable:         monthlycharges   R-squared:                       0.079
Model:                            OLS   Adj. R-squared:                  0.079
Method:                 Least Squares   F-statistic:                     200.9
Date:                Tue, 24 Feb 2026   Prob (F-statistic):          4.81e-125
Time:                        15:49:04   Log-Likelihood:                -33625.
No. Observations:                7032   AIC:                         6.726e+04
Df Residuals:                    7028   BIC:                         6.729e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               50.9211 

## 第三关（实战审判关）：读懂体检报告
请观察第一关你跑出的 OLS Summary 报告，不要写代码，直接找出关键指标并答复：

1. **分类变量提取与除草**：在你的模型里，`gender` (性别) 显著影响月费吗？你的依据是表里的哪个字段？这个字段的值必须小于多少我们才能采信它具有显著影响？
   1. 答：不显著，P值0.233，必须要小于0.05才算显著
2. **宏观解释力**：整个预测模型的 **调整 R 方 (Adj. R-squared)** 大约是多少？这意味着这三个基础变量在数学上能多大程度解释大家话费高低的不同？
   1. 0.085，只解释了8.5%的差异

✅ *做完后，你可以直接向下翻看参考答案。*

---
### 💡 参考答案

In [6]:
# 第一关解答
formula_1 = 'monthlycharges ~ C(gender) + C(dependents) + tenure'
model_1 = smf.ols(formula_1, data=df).fit()
print(model_1.summary())

# 审判关解答：
# 1. (不显著)。从表中找到 C(gender)[T.Male] 那一行的 `P>|t|` (P-value)，如果它的值(比如大概 0.5+)远超 0.05 的显著性阈值，说明性别在当前模型下没有起到造成话费高低的区分度，是噪音而已。
# 2. R-squared 约 0.063。即性别、长居人口和在网时间仅仅能解释用户 6.3% 的话费差距。剩下的 93.7% 是未解释的（也就是上一篇里的，比如用不用光纤网络，买没买流媒体服务！这也是为什么我们之前遗漏变量会导致 49.88 离谱偏差的深层原因！）

# 第二关解答 (直接在公式中使用 Numpy)
formula_2 = 'monthlycharges ~ C(gender) + C(dependents) + np.log(tenure)'
model_2 = smf.ols(formula_2, data=df).fit()
print(model_2.summary())

                            OLS Regression Results                            
Dep. Variable:         monthlycharges   R-squared:                       0.085
Model:                            OLS   Adj. R-squared:                  0.085
Method:                 Least Squares   F-statistic:                     217.8
Date:                Tue, 24 Feb 2026   Prob (F-statistic):          3.98e-135
Time:                        17:08:25   Log-Likelihood:                -33602.
No. Observations:                7032   AIC:                         6.721e+04
Df Residuals:                    7028   BIC:                         6.724e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               57.4559 